# 02 Truncation Depth Ablation

This notebook visualizes the truncation-depth ablation.

- Input: `result/table/02_truncation_depth_ablation.csv`
- Outputs: PDF figures in `result/figure/` and aggregated tables in `result/table/`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path(__file__).resolve().parents[1]
tbl_dir = ROOT / "result" / "table"
fig_dir = ROOT / "result" / "figure"
fig_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(tbl_dir / "02_truncation_depth_ablation.csv")
df.head()


In [ ]:
# Aggregate mean/std across runs for each (neumann_steps, lower_steps_per_slot)
metrics = ["avg_sum_queue", "avg_sense_u", "alpha_total_variation", "avg_slot_ms", "deadline_violation_rate"]

agg = df.groupby(["neumann_steps", "lower_steps_per_slot"])[metrics].agg(["mean", "std"]).reset_index()
agg.columns = ["neumann_steps", "lower_steps_per_slot"] + [f"{m}_{s}" for m in metrics for s in ["mean", "std"]]

out_table = tbl_dir / "02_truncation_depth_table.csv"
agg.to_csv(out_table, index=False)
print("Saved table:", out_table)

agg.head()


In [ ]:
def heatmap(metric: str, fname: str, title: str):
    # Pivot for heatmap
    piv = agg.pivot(index="lower_steps_per_slot", columns="neumann_steps", values=f"{metric}_mean")
    plt.figure()
    im = plt.imshow(piv.values, aspect="auto", origin="lower")
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.xticks(range(len(piv.columns)), piv.columns)
    plt.yticks(range(len(piv.index)), piv.index)
    plt.xlabel("Neumann steps")
    plt.ylabel("Follower steps per slot")
    plt.title(title)
    out = fig_dir / fname
    plt.savefig(out, format="pdf", bbox_inches="tight")
    print("Saved:", out)

heatmap("avg_sum_queue", "02_heatmap_avg_queue.pdf", "Average total queue (mean)")
heatmap("avg_sense_u", "02_heatmap_avg_sense_u.pdf", "Average sensing uncertainty (mean)")
heatmap("avg_slot_ms", "02_heatmap_avg_slot_ms.pdf", "Average per-slot runtime (ms, mean)")


In [ ]:
# Runtime lines: for each follower-step budget, runtime vs Neumann steps
plt.figure()
for ls in sorted(agg["lower_steps_per_slot"].unique()):
    sub = agg[agg["lower_steps_per_slot"] == ls].sort_values("neumann_steps")
    plt.plot(sub["neumann_steps"], sub["avg_slot_ms_mean"], marker="o", label=f"ls={ls}")
plt.xlabel("Neumann steps")
plt.ylabel("Average per-slot runtime (ms)")
plt.legend()
plt.grid(True, alpha=0.3)
out = fig_dir / "02_runtime_vs_neumann_steps.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out)


In [ ]:
# Efficiency scatter: runtime vs queue and runtime vs sensing
plt.figure()
plt.scatter(agg["avg_slot_ms_mean"], agg["avg_sum_queue_mean"])
plt.xlabel("Avg per-slot runtime (ms)")
plt.ylabel("Average total queue")
plt.grid(True, alpha=0.3)
out = fig_dir / "02_scatter_runtime_vs_queue.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out)

plt.figure()
plt.scatter(agg["avg_slot_ms_mean"], agg["avg_sense_u_mean"])
plt.xlabel("Avg per-slot runtime (ms)")
plt.ylabel("Average sensing uncertainty $u_t$")
plt.grid(True, alpha=0.3)
out = fig_dir / "02_scatter_runtime_vs_sense_u.pdf"
plt.savefig(out, format="pdf", bbox_inches="tight")
print("Saved:", out)
